# Data Quality Assessment — Standalone Notebook

All pipeline logic is defined **inline** in this notebook.  
No ml-server source code required — runs anywhere with:
```
pip install pydantic numpy scipy pandas requests plotly
```

Sections:
1. **Load pipeline module**
2. **Batch assessment** — reads `inputs.csv` + `models.csv`, runs pipeline for every archive
3. **Anomaly visualization** — select one result and explore dashboard, timeline, actions, per-type zoom
4. **Export** — CSV + JSON


In [1]:
import os, sys, json, pathlib, math, time, logging
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import requests
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from IPython.display import display

# SCADA / batch settings
SCADA_URL  = os.getenv('SCADA_URL', 'http://127.0.0.1:7080/api/v1/read/archives')
OBJECT_REF = os.getenv('OBJECT_REF', None)
STEP       = int(os.getenv('DQ_STEP', '3600'))

# Pipeline tuning
ALLOW_LOOK_AHEAD  = True
Z_SCORE_WINDOW    = 48
Z_SCORE_THRESHOLD = 3.0
STUCK_WINDOW      = 10

print('Standalone notebook loaded.')
print(f'SCADA URL : {SCADA_URL}')
print(f'Archive   : {OBJECT_REF}')
print(f'Step      : {STEP}s')

Standalone notebook loaded.
SCADA URL : http://127.0.0.1:7080/api/v1/read/archives
Archive   : None
Step      : 3600s


---
## 1 — Load Pipeline Module

Locates `lib/pipeline.py` in the ml-server directory tree and imports all classes.
No ml-server FastAPI stack needed — only `pydantic`, `numpy`, `scipy`.

In [2]:
import sys, pathlib

def _find_lib():
    current = pathlib.Path(os.path.abspath('.'))
    for _ in range(10):
        cand = current / 'lib' / 'pipeline.py'
        if cand.exists():
            lib_dir = str(current)
            if lib_dir not in sys.path:
                sys.path.insert(0, lib_dir)
            return lib_dir
        current = current.parent
    return None

_lib_root = _find_lib()
if _lib_root is None:
    raise ImportError(
        'Cannot locate lib/pipeline.py.  '
        'Run this notebook from inside the ml-server directory tree.')

from lib.pipeline import (  # noqa: E402
    _parse_time, _ts_to_iso, _dt_to_ms, _find_nan_runs,
    _step1_chronological, _step2_static_bounds, _step3_dynamic,
    _step4_impute, _compute_score,
    ScoringWeights, AssessRequest, AnomalyRecord, TagStats,
    MetricsScoring, Metadata, AssessResponse,
    DataQualityPipeline,
)

print(f'lib/pipeline.py loaded from: {_lib_root}')
print('  AssessRequest, DataQualityPipeline, AssessResponse -- all ready.')

lib/pipeline.py loaded from: /Users/rustamkrikbayev/Documents/projects/forecast/ml-server
  AssessRequest, DataQualityPipeline, AssessResponse -- all ready.


---
## 2 — Batch Assessment from `inputs.csv`

Reads `local/models/training_workspace/models_enabled/inputs.csv` and `models.csv`,
joins on `object_ref`, fetches each archive from SCADA directly, and runs the inline pipeline.

| Column | Source | Meaning |
|--------|--------|---------|
| `input_ref` | inputs.csv | SCADA archive path to assess |
| `api_url` | inputs.csv | SCADA endpoint for that archive |
| `object_ref` | inputs.csv | Model reference (join key) |
| `step` | models.csv | Time step in seconds |
| `input_range` | models.csv | History window in steps |

**No server needed** -- fetches SCADA directly and runs the pipeline in-process.

In [3]:
def _find_enabled_dir():
    current = pathlib.Path(os.path.abspath('.'))
    for _ in range(10):
        cand = current / 'local' / 'models' / 'training_workspace' / 'models_enabled'
        if cand.is_dir():
            return cand
        cand2 = current / 'models' / 'training_workspace' / 'models_enabled'
        if cand2.is_dir():
            return cand2
        current = current.parent
    return None

ENABLED_DIR = _find_enabled_dir()
if ENABLED_DIR is None:
    raise FileNotFoundError(
        'Cannot locate models_enabled/ directory. '
        'Set ENABLED_DIR = pathlib.Path("/absolute/path") below.')

# Override here if needed:
# ENABLED_DIR = pathlib.Path('/absolute/path/to/models_enabled')

INPUTS_CSV = ENABLED_DIR / 'inputs.csv'
MODELS_CSV = ENABLED_DIR / 'models.csv'

BATCH_HOURS         = None   # None -> use input_range * step; or e.g. 168
BATCH_SCADA_TIMEOUT = 30     # seconds per archive

print(f'inputs.csv : {INPUTS_CSV}  (exists={INPUTS_CSV.exists()})')
print(f'models.csv : {MODELS_CSV}  (exists={MODELS_CSV.exists()})')

inputs.csv : /Users/rustamkrikbayev/Documents/projects/forecast/local/models/training_workspace/models_enabled/inputs.csv  (exists=True)
models.csv : /Users/rustamkrikbayev/Documents/projects/forecast/local/models/training_workspace/models_enabled/models.csv  (exists=True)


In [4]:
df_inputs = pd.read_csv(INPUTS_CSV, sep=';', comment='#',
                         names=['row_id','object_ref','pattern','input_ref','api_url'],
                         dtype=str).dropna(subset=['input_ref'])

df_models = pd.read_csv(MODELS_CSV, sep=';', comment='#',
                         names=['row_id','object_ref','input_range','output_range','step'],
                         dtype=str).dropna(subset=['object_ref'])
df_models['step']        = pd.to_numeric(df_models['step'],        errors='coerce').fillna(3600).astype(int)
df_models['input_range'] = pd.to_numeric(df_models['input_range'], errors='coerce').fillna(168).astype(int)

df_batch = (df_inputs
    .merge(df_models[['object_ref','step','input_range']], on='object_ref', how='left')
    .assign(step=lambda d: d['step'].fillna(3600).astype(int),
            input_range=lambda d: d['input_range'].fillna(168).astype(int))
    .reset_index(drop=True))

print(f'Loaded {len(df_batch)} input(s) to assess:')
display(df_batch[['row_id','object_ref','input_ref','step','input_range','api_url']])

Loaded 2 input(s) to assess:


,row_id,object_ref,input_ref,step,input_range,api_url
0,1214,/root/FP/PROJECT/AKMOLA/@regions/North Kazakhs...,/root/FP/PROJECT/AKMOLA/@regions/SevKaz/Load/P...,3600,360,http://127.0.0.1:7080/api/v1/read/archives
1,1170,/root/FP/PROJECT/AKMOLA/Nura_SES/@models/P_watt,/root/FP/PROJECT/AKMOLA/Nura_SES/Pgen_sum/arch...,3600,168,http://127.0.0.1:7080/api/v1/read/archives


In [5]:
from urllib.parse import urlparse, urlunparse

def _normalize_scada_url(url):
    parsed = urlparse(url)
    if parsed.hostname not in ('127.0.0.1', 'localhost'):
        return url
    if not pathlib.Path('/.dockerenv').exists():
        return url
    netloc = parsed.netloc.replace(parsed.hostname, 'host.docker.internal')
    return urlunparse(parsed._replace(netloc=netloc))


def _fetch_scada(api_url, archive, from_ms, to_ms, step_s, timeout=30):
    url  = _normalize_scada_url(api_url)
    body = {'from': from_ms, 'to': to_ms, 'archive': [archive], 'step': step_s}
    try:
        resp = requests.post(url, json=body, timeout=timeout)
        if resp.status_code == 200:
            data = resp.json()
            if isinstance(data, dict) and data:
                return data
    except Exception:
        pass
    return None


now_utc       = datetime.now(tz=timezone.utc).replace(minute=0, second=0, microsecond=0)
BATCH_RESULTS = []

for _, row in df_batch.iterrows():
    object_ref  = str(row['object_ref']).strip()
    input_ref   = str(row['input_ref']).strip()
    api_url     = str(row['api_url']).strip()
    step_s      = int(row['step'])
    input_range = int(row['input_range'])

    window_h = BATCH_HOURS if BATCH_HOURS else (input_range * step_s) // 3600
    to_dt    = now_utc
    from_dt  = to_dt - timedelta(hours=window_h)
    from_ms  = int(from_dt.timestamp() * 1000)
    to_ms    = int(to_dt.timestamp() * 1000)

    arch_ref_display = input_ref.replace('/root/FP/PROJECT/', '') if '/root/FP/PROJECT/' in input_ref else input_ref
    model_ref_display = object_ref.replace('/root/FP/PROJECT/', '') if '/root/FP/PROJECT/' in object_ref else object_ref
    short_model = object_ref.split('/')[-1] if '/' in object_ref else object_ref

    print(f'[{model_ref_display}]  arch={arch_ref_display}  window={window_h}h  step={step_s}s', end='  ')

    entry = dict(
        object_ref=object_ref, input_ref=input_ref,
        short_model=short_model, short_input=arch_ref_display,
        step_s=step_s, window_h=window_h,
        from_dt=from_dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
        to_dt=to_dt.strftime('%Y-%m-%dT%H:%M:%SZ'),
        status='ok', score=None,
        n_expected=None, n_missing=None, n_dup=None,
        n_spikes=None, n_stuck=None, n_roc=None, n_long_gaps=None,
        n_raw_points=None, error=None, pipeline_result=None,
    )

    raw_payload = _fetch_scada(api_url, input_ref, from_ms, to_ms, step_s,
                               timeout=BATCH_SCADA_TIMEOUT)
    if raw_payload is None:
        entry.update(status='scada_unavailable', error=f'SCADA did not respond: {api_url}')
        print('SCADA unavailable')
        BATCH_RESULTS.append(entry)
        continue

    raw_series = raw_payload.get(input_ref, []) or next(iter(raw_payload.values()), [])
    entry['n_raw_points'] = len(raw_series)
    if not raw_series:
        entry.update(status='no_data', error='SCADA returned empty series')
        print('no data')
        BATCH_RESULTS.append(entry)
        continue

    try:
        req = AssessRequest(
            **{'from': entry['from_dt']},
            object_ref=input_ref, to=entry['to_dt'], step=step_s,
            allow_look_ahead=ALLOW_LOOK_AHEAD,
            z_score_window=Z_SCORE_WINDOW,
            z_score_threshold=Z_SCORE_THRESHOLD,
            stuck_window=STUCK_WINDOW,
        )
        res = DataQualityPipeline(req).run({input_ref: raw_series})
        st  = res.metrics_scoring.tags[input_ref]
        entry.update(
            score=res.metrics_scoring.overall_quality_score,
            n_expected=st.total_expected_points,
            n_missing=st.missing_points_count, n_dup=st.duplicates_count,
            n_spikes=st.outliers_count, n_stuck=st.stuck_sequences_count,
            n_roc=st.rate_of_change_count, n_long_gaps=st.long_gaps_count,
            pipeline_result=res,
        )
        badge = 'OK' if entry['score'] >= 80 else 'WARN' if entry['score'] >= 60 else 'LOW'
        print(f'{badge} score={entry["score"]:.1f}  miss={st.missing_points_count}')
    except Exception as exc:
        entry.update(status='pipeline_error', error=str(exc))
        print(f'ERROR: {exc}')

    BATCH_RESULTS.append(entry)

print(f'\nDone: {len(BATCH_RESULTS)} inputs assessed.')

[AKMOLA/@regions/North Kazakhstan/load/@models/P_watt]  arch=AKMOLA/@regions/SevKaz/Load/P_Load/archives/out_value  window=360h  step=3600s  OK score=99.0  miss=2
[AKMOLA/Nura_SES/@models/P_watt]  arch=AKMOLA/Nura_SES/Pgen_sum/archives/out_value  window=168h  step=3600s  OK score=98.2  miss=3

Done: 2 inputs assessed.


In [6]:
df_summary = pd.DataFrame([{
    'Model':      r['short_model'], 'Archive':    r['short_input'],
    'Step':       f"{r['step_s']}s", 'Window':  f"{r['window_h']}h",
    'Status':     r['status'],       'Score':     r['score'],
    'Expected':   r['n_expected'],   'Raw pts':   r['n_raw_points'],
    'Missing':    r['n_missing'],    'Duplicates': r['n_dup'],
    'Spikes':     r['n_spikes'],     'Stuck':     r['n_stuck'],
    'RoC':        r['n_roc'],        'Long gaps': r['n_long_gaps'],
    'Error':      r['error'],
} for r in BATCH_RESULTS])

def _color_score(val):
    if pd.isna(val):    return 'background-color: #f5b7b1'
    if val >= 80:       return 'background-color: #d5f5e3'
    if val >= 60:       return 'background-color: #fdebd0'
    return                     'background-color: #f5b7b1'

print('Batch quality assessment summary:')
display(df_summary.style
    .applymap(_color_score, subset=['Score'])
    .format({'Score': lambda v: f'{v:.1f}' if pd.notna(v) else '--'})
    .set_properties(**{'text-align': 'right'}))

Batch quality assessment summary:


/var/folders/z9/_xmzk2xd65b6z8w_nqdgdbkm0000gn/T/ipykernel_4450/1968626056.py:20: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(_color_score, subset=['Score'])


,Model,Archive,Step,Window,Status,Score,Expected,Raw pts,Missing,Duplicates,Spikes,Stuck,RoC,Long gaps,Error
0,P_watt,AKMOLA/@regions/SevKaz/Load/P_Load/archives/out_value,3600s,360h,ok,99.0,361,360,2,0,2,0,1,0,None
1,P_watt,AKMOLA/Nura_SES/Pgen_sum/archives/out_value,3600s,168h,ok,98.2,169,168,3,0,0,0,1,0,None


In [7]:
ok_rows = [r for r in BATCH_RESULTS if r['score'] is not None]

if not ok_rows:
    print('No successful assessments to visualise.')
else:
    labels  = [f"{r['short_model']}\n{r['short_input']}" for r in ok_rows]
    scores  = [r['score'] for r in ok_rows]
    colors  = ['#2ecc71' if s >= 80 else '#e67e22' if s >= 60 else '#e74c3c' for s in scores]

    fig_bar = go.Figure(go.Bar(x=labels, y=scores, marker_color=colors,
        text=[f'{s:.1f}' for s in scores], textposition='outside'))
    fig_bar.add_hline(y=80, line_dash='dash', line_color='#e74c3c',
                      annotation_text='threshold 80', annotation_position='top right')
    fig_bar.update_layout(title='Input Data Quality -- All Models',
        yaxis=dict(title='Score / 100', range=[0, 115]),
        xaxis_title='Model / Archive', template='plotly_white', height=420)
    fig_bar.show()

    defect_cols   = ['Missing', 'Duplicates', 'Spikes', 'Stuck', 'RoC']
    defect_keys   = ['n_missing', 'n_dup', 'n_spikes', 'n_stuck', 'n_roc']
    defect_colors = ['#3498db', '#9b59b6', '#e67e22', '#f39c12', '#1abc9c']
    fig_def = go.Figure()
    for col, key, color in zip(defect_cols, defect_keys, defect_colors):
        fig_def.add_trace(go.Bar(name=col, x=labels,
            y=[r.get(key) or 0 for r in ok_rows], marker_color=color))
    fig_def.update_layout(barmode='stack', title='Defect Breakdown -- All Models',
        xaxis_title='Model / Archive', yaxis_title='Point count',
        template='plotly_white', height=400)
    fig_def.show()

    completeness = [
        round(100 * (r['n_raw_points'] or 0) / r['n_expected'], 1)
        if r['n_expected'] else None
        for r in ok_rows
    ]
    fig_comp = go.Figure(go.Bar(x=labels, y=completeness,
        marker_color=['#2ecc71' if (v or 0) >= 95 else '#e67e22' if (v or 0) >= 80 else '#e74c3c'
                      for v in completeness],
        text=[f'{v}%' if v is not None else '--' for v in completeness], textposition='outside'))
    fig_comp.add_hline(y=95, line_dash='dash', line_color='#e74c3c',
                       annotation_text='95%', annotation_position='top right')
    fig_comp.update_layout(title='Raw Data Completeness (received vs expected)',
        yaxis=dict(title='%', range=[0, 115]),
        xaxis_title='Model / Archive', template='plotly_white', height=380)
    fig_comp.show()

In [8]:
ok_rows = [r for r in BATCH_RESULTS if r['pipeline_result'] is not None]

if not ok_rows:
    print('No pipeline results to plot.')
else:
    pal = ['#2ecc71','#e74c3c','#9b59b6','#f39c12','#1abc9c',
           '#3498db','#e67e22','#1a5276','#7d3c98','#117a65']
    fig_all = go.Figure()
    for i, r in enumerate(ok_rows):
        res   = r['pipeline_result']
        tag   = r['input_ref']
        df_cl = pd.DataFrame(res.cleaned_data)
        df_cl['ts'] = pd.to_datetime(df_cl['timestamp'], utc=True).dt.tz_convert(None)
        if tag not in df_cl.columns:
            continue
        fig_all.add_trace(go.Scatter(
            x=df_cl['ts'], y=df_cl[tag].astype(float), mode='lines',
            name=f"{r['short_model']} ({r['score']:.0f}/100)",
            line=dict(color=pal[i % len(pal)], width=1.6)))
    fig_all.update_layout(title='Cleaned Input Series -- All Archives',
        xaxis_title='Time (UTC)', yaxis_title='Value',
        hovermode='x unified', template='plotly_white', height=500,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
    fig_all.show()

---
## 3 — Anomaly Visualization

Select one result from `BATCH_RESULTS` by index and explore four views:

1. **Dashboard** — quality score gauge + per-type anomaly counts
2. **Timeline** — when each anomaly type occurs across the window
3. **Actions taken** — what the pipeline did with each anomaly
4. **Per-type zoom** — cleaned series with anomalies of each type highlighted

In [13]:
# ── 3.0  Pick a result to visualise ────────────────────────────────────────
# Change RESULT_IDX to explore a different archive.
RESULT_IDX = 0

ok_results = [r for r in BATCH_RESULTS if r.get('pipeline_result') is not None]
if not ok_results:
    raise RuntimeError("No successful pipeline results to visualise. Run Section 2 first.")

_r    = ok_results[RESULT_IDX]
result = _r['pipeline_result']          # AssessResponse object
TAG   = next(iter(result.metrics_scoring.tags))  # first (and usually only) tag key

print(f"Visualising: {_r['short_model']} / {_r['short_input']}")
print(f"  Tag key : {TAG}")
print(f"  Score   : {result.metrics_scoring.overall_quality_score:.2f}/100")
print(f"  Anomalies: {len(result.anomalies_log)}")


Visualising: P_watt / AKMOLA/@regions/SevKaz/Load/P_Load/archives/out_value
  Tag key : /root/FP/PROJECT/AKMOLA/@regions/SevKaz/Load/P_Load/archives/out_value
  Score   : 99.00/100
  Anomalies: 8


In [14]:
# ── 3.1  Score gauge + anomaly count bars ─────────────────────────────────
score = result.metrics_scoring.overall_quality_score
st    = result.metrics_scoring.tags[TAG]

count_labels = ['Missing', 'Duplicates', 'Spikes', 'Stuck', 'RoC', 'Long gaps']
count_values = [
    st.missing_points_count, st.duplicates_count,
    st.outliers_count, st.stuck_sequences_count,
    st.rate_of_change_count, st.long_gaps_count,
]
count_colors = ['#3498db', '#9b59b6', '#e67e22', '#f39c12', '#1abc9c', '#e74c3c']

fig_dash = make_subplots(
    rows=1, cols=1,
    specs=[[{'type': 'bar'}]],
    subplot_titles=('Anomaly counts by type'),
    column_widths=[0.65],
)

# fig_dash.add_trace(go.Indicator(
#     mode='gauge+number',
#     value=score,
#     gauge={
#         'axis': {'range': [0, 100]},
#         'bar': {'color': '#2ecc71' if score >= 80 else '#e67e22' if score >= 60 else '#e74c3c'},
#         'steps': [
#             {'range': [0,  60], 'color': '#fadbd8'},
#             {'range': [60, 80], 'color': '#fdebd0'},
#             {'range': [80, 100], 'color': '#d5f5e3'},
#         ],
#         'threshold': {'line': {'color': 'red', 'width': 3}, 'thickness': 0.75, 'value': 80},
#     },
#     number={'suffix': ' / 100'},
# ), row=1, col=1)

fig_dash.add_trace(go.Bar(
    x=count_labels, y=count_values,
    marker_color=count_colors,
    text=count_values, textposition='outside',
    showlegend=False,
), row=1, col=1)

fig_dash.update_layout(
    title=f'Data Quality Dashboard  |  {_r["short_model"]}  |  Score: {score:.1f}/100',
    template='plotly_white', height=380,
    yaxis2=dict(title='point count'),
)
fig_dash.show()


In [15]:
# ── 3.2  Anomaly timeline (type × time) ───────────────────────────────────
ANOM_COLORS = {
    'missing': '#3498db', 'duplicate': '#9b59b6', 'hard_limit': '#e74c3c',
    'spike_outlier': '#e67e22', 'stuck_signal': '#f39c12', 'rate_of_change': '#1abc9c',
}
TYPE_ORDER = ['missing', 'duplicate', 'hard_limit', 'spike_outlier', 'stuck_signal', 'rate_of_change']

df_a = pd.DataFrame([a.model_dump() for a in result.anomalies_log])
if df_a.empty:
    print("No anomalies in this result — nothing to plot.")
else:
    df_a['ts']    = pd.to_datetime(df_a['timestamp'], utc=True).dt.tz_convert(None)
    df_a['y_pos'] = df_a['anomaly_type'].map({t: i for i, t in enumerate(TYPE_ORDER)})
    df_a['size']  = df_a['raw_value'].notna().map({True: 10, False: 6})

    fig_tl = go.Figure()
    for atype in TYPE_ORDER:
        sub = df_a[df_a['anomaly_type'] == atype]
        if sub.empty:
            continue
        hover = [
            f"{atype}<br>{row.ts.strftime('%Y-%m-%d %H:%M')}<br>"
            f"raw={row.raw_value:.2f}<br>action={row.action_taken}"
            if pd.notna(row.raw_value) else
            f"{atype}<br>{row.ts.strftime('%Y-%m-%d %H:%M')}<br>action={row.action_taken}"
            for _, row in sub.iterrows()
        ]
        fig_tl.add_trace(go.Scatter(
            x=sub['ts'], y=sub['y_pos'],
            mode='markers', name=atype,
            marker=dict(color=ANOM_COLORS.get(atype, '#95a5a6'), size=sub['size'].tolist(),
                        opacity=0.85, line=dict(width=0.5, color='white')),
            hovertext=hover, hoverinfo='text',
        ))

    present     = [t for t in TYPE_ORDER if t in df_a['anomaly_type'].values]
    present_idx = [TYPE_ORDER.index(t) for t in present]
    fig_tl.update_layout(
        title=f'Anomaly Timeline  |  {_r["short_model"]}',
        xaxis_title='Time (UTC)',
        yaxis=dict(tickvals=present_idx, ticktext=present, autorange='reversed'),
        template='plotly_white', height=380,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )
    fig_tl.show()


In [16]:
# ── 3.3  Action taken distribution ────────────────────────────────────────
if df_a.empty:
    print("No anomalies — skipping action chart.")
else:
    action_counts = df_a['action_taken'].value_counts()
    ACTION_COLORS = {
        'linear_interpolated':   '#2ecc71',
        'cubic_interpolated':    '#27ae60',
        'forward_filled':        '#3498db',
        'deduplicated':          '#9b59b6',
        'replaced_with_nan':     '#e74c3c',
        'marked_as_nan_no_fill': '#c0392b',
        'long_gap_not_filled':   '#e67e22',
    }
    colors = [ACTION_COLORS.get(a, '#95a5a6') for a in action_counts.index]

    fig_act = go.Figure(go.Pie(
        labels=action_counts.index.tolist(),
        values=action_counts.values.tolist(),
        marker_colors=colors,
        hole=0.45,
        textinfo='label+percent+value',
        hoverinfo='label+value',
    ))
    fig_act.update_layout(
        title=f'Actions Taken by Pipeline  |  {_r["short_model"]}',
        template='plotly_white', height=420,
    )
    fig_act.show()


In [17]:
# ── 3.4  Per-type zoom: cleaned series with highlighted anomalies ──────────
if df_a.empty:
    print("No anomalies — nothing to zoom into.")
else:
    df_cl = pd.DataFrame(result.cleaned_data)
    df_cl['ts']    = pd.to_datetime(df_cl['timestamp'], utc=True).dt.tz_convert(None)
    df_cl['value'] = df_cl[TAG].astype(float)

    types_present = [t for t in TYPE_ORDER if t in df_a['anomaly_type'].values]
    n_plots = len(types_present)

    fig_zoom = make_subplots(
        rows=n_plots, cols=1,
        shared_xaxes=True,
        subplot_titles=[
            f'{t} ({len(df_a[df_a["anomaly_type"]==t])} events)' for t in types_present
        ],
        vertical_spacing=0.06,
    )

    for row_i, atype in enumerate(types_present, start=1):
        color = ANOM_COLORS.get(atype, '#95a5a6')
        sub   = df_a[df_a['anomaly_type'] == atype].copy()

        t_min  = sub['ts'].min() - pd.Timedelta(hours=4)
        t_max  = sub['ts'].max() + pd.Timedelta(hours=4)
        df_win = df_cl[(df_cl['ts'] >= t_min) & (df_cl['ts'] <= t_max)]

        fig_zoom.add_trace(go.Scatter(
            x=df_win['ts'], y=df_win['value'],
            mode='lines', name='Cleaned',
            line=dict(color='#bdc3c7', width=1.2),
            showlegend=(row_i == 1),
        ), row=row_i, col=1)

        marker_y = sub.merge(df_cl[['ts', 'value']], on='ts', how='left')['value']
        hover_txt = [
            f"<b>{atype}</b><br>"
            f"{r.ts.strftime('%Y-%m-%d %H:%M')}<br>"
            f"raw={r.raw_value:.3f}<br>action={r.action_taken}"
            if pd.notna(r.raw_value) else
            f"<b>{atype}</b><br>{r.ts.strftime('%Y-%m-%d %H:%M')}<br>action={r.action_taken}"
            for _, r in sub.iterrows()
        ]
        fig_zoom.add_trace(go.Scatter(
            x=sub['ts'], y=marker_y,
            mode='markers', name=atype,
            marker=dict(color=color, size=10, symbol='x-open', line=dict(width=2)),
            hovertext=hover_txt, hoverinfo='text',
            showlegend=(row_i == 1),
        ), row=row_i, col=1)

    fig_zoom.update_layout(
        title=f'Per-Type Anomaly Zoom  |  {_r["short_model"]}',
        hovermode='x unified',
        template='plotly_white',
        height=240 * n_plots,
        legend=dict(orientation='h', yanchor='bottom', y=1.01, xanchor='right', x=1),
    )
    fig_zoom.show()

    display(
        df_a[['ts', 'anomaly_type', 'action_taken', 'raw_value', 'duration_seconds']]
        .sort_values(['anomaly_type', 'ts'])
        .reset_index(drop=True)
    )


,ts,anomaly_type,action_taken,raw_value,duration_seconds
0,2026-05-18 14:00:00,missing,linear_interpolated,NaN,None
1,2026-05-18 18:00:00,missing,linear_interpolated,NaN,None
2,2026-05-28 04:00:00,missing,linear_interpolated,NaN,None
3,2026-05-28 20:00:00,missing,linear_interpolated,NaN,None
4,2026-05-31 17:00:00,missing,linear_interpolated,NaN,None
5,2026-05-18 18:00:00,rate_of_change,replaced_with_nan,352.560479,None
6,2026-05-18 14:00:00,spike_outlier,replaced_with_nan,426.342950,None
7,2026-05-28 20:00:00,spike_outlier,replaced_with_nan,-168.016647,None


In [ ]:
export_dir = pathlib.Path('.') / 'exports'
export_dir.mkdir(exist_ok=True)
ts_now = datetime.now().strftime('%Y%m%d_%H%M%S')

# fpath_summary = export_dir / f'batch_quality_summary__{ts_now}.csv'
# df_summary.to_csv(fpath_summary, index=False)
# print(f'Summary CSV   -> {fpath_summary}')

for r in BATCH_RESULTS:
    if r['pipeline_result'] is None:
        continue
    alog = r['pipeline_result'].anomalies_log
    if not alog:
        continue
    slug    = r['short_input'][:40].replace(' ', '_')
    fpath_a = export_dir / f'batch_anomalies__{slug}__{ts_now}.csv'
    os.makedirs(fpath_a.parent, exist_ok=True)
    pd.DataFrame([a.model_dump() for a in alog]).to_csv(fpath_a, index=False)
    print(f'Anomaly log   -> {fpath_a}  ({len(alog)} records)')

batch_report = [{
    'object_ref': r['object_ref'], 'input_ref': r['input_ref'],
    'from': r['from_dt'], 'to': r['to_dt'], 'step_s': r['step_s'],
    'status': r['status'], 'score': r['score'],
    'stats': {k: r[k] for k in ('n_expected','n_raw_points','n_missing',
                                  'n_dup','n_spikes','n_stuck','n_roc','n_long_gaps')},
    'error': r['error'],
} for r in BATCH_RESULTS]

fpath_json = export_dir / f'batch_quality_report__{ts_now}.json'
with open(fpath_json, 'w') as f:
    json.dump(batch_report, f, indent=2, ensure_ascii=False, default=str)
print(f'Batch report  -> {fpath_json}')

Anomaly log   -> exports/batch_anomalies__AKMOLA/@regions/SevKaz/Load/P_Load/archi__20260531_215914.csv  (8 records)
Anomaly log   -> exports/batch_anomalies__AKMOLA/Nura_SES/Pgen_sum/archives/out_va__20260531_215914.csv  (5 records)
Batch report  -> exports/batch_quality_report__20260531_215914.json
